# Sentinel-1 SLC → gamma0 VV/VH, orthorectified on a common grid

Takes four SLC products and an AOI, keeps only the bursts the AOI touches, and
produces one GeoTIFF per date with two named bands (`gamma0_VV`, `gamma0_VH`).

**Pipeline**, per product / sub-swath / polarisation:
`SARBurstExtraction` → `SARConcatenateBursts` → `SARCalibration (gamma)` →
`OrthoRectification`, then the sub-swaths are merged and written as a 2-band file.

**Coregistration** is geometric: every date is orthorectified onto the *identical*
grid (same EPSG, same origin, same size, same DEM), so the outputs stack pixel to
pixel. This is not InSAR coregistration — OTB does not do that.

**Two limits to keep in mind.** OTB's gamma0 comes from the annotation LUT, on the
ellipsoid: it is *not* terrain-flattened (no equivalent of SNAP's Terrain
Flattening). And OTB uses the orbit vectors embedded in the product, not the
downloaded precise orbits.

**To validate before trusting the results** (cell 4 helps): OTB application names
and parameter keys vary between versions, and `SARBurstExtraction` burst indices
are assumed 0-based here while `polygon_to_swaths_bursts` returns 1-based numbers.

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import rasterio
from rasterio.transform import from_origin

# polygon_to_swaths_bursts is a sibling folder: make it importable
TOOLS = Path.cwd().parent / "polygon_to_swaths_bursts"
sys.path.insert(0, str(TOOLS))
from polygon_to_swaths_bursts import get_intersecting_bursts, parse_polygon

In [ ]:
# --- Parameters: adapt to your data ---

# The four SLC products. OTB reads extracted .SAFE directories, not .zip archives.
SLC_PATHS = [
    "C:/Users/guigu/Documents/pro_asus/vigisar/data/data_raw/zta10/S1B_IW_SLC__1SDV_20170804T215105_20170804T215131_006796_00BF5A_B333.SAFE",
    "C:/path/to/second.SAFE",
    "C:/path/to/third.SAFE",
    "C:/path/to/fourth.SAFE",
]

# Area of interest, lon/lat (EPSG:4326): inline WKT, or a WKT / GeoJSON file path
AOI = "POLYGON ((-54.383019 5.252325, -54.546861 5.252325, -54.546861 5.128303, -54.383019 5.128303, -54.383019 5.252325))"

# DEM for orthorectification: directory holding the SRTM tiles, and the geoid file
DEM_DIR = "C:/path/to/srtm"
GEOID = "C:/path/to/egm96.grd"

OUT_DIR = "output"
TMP_DIR = "tmp"

PIXEL_SIZE = 10.0            # metres, output grid
POLARISATIONS = ["VV", "VH"]
COARSE = True                # burst selection: favour recall near seams

In [ ]:
# --- Which swaths and bursts does the AOI touch, in each product? ---
selection = {}
for slc in SLC_PATHS:
    _, summary = get_intersecting_bursts(slc, AOI, coarse=COARSE)
    selection[slc] = summary
    print(Path(slc).name)
    for swath, bursts in sorted(summary.items()):
        print(f"    {swath}: bursts {bursts}")

# Products on the same relative orbit should hit the same sub-swaths. A mismatch
# means a different track: the outputs would still align on the grid, but the
# viewing geometry differs and the series is no longer directly comparable.
if len({frozenset(s) for s in selection.values()}) > 1:
    print("\nWARNING: the products do not cover the same sub-swaths.")

In [ ]:
# --- OTB availability, and the exact parameter keys of this OTB version ---
import otbApplication as otb

NEEDED = [
    "SARBurstExtraction",
    "SARConcatenateBursts",
    "SARCalibration",
    "OrthoRectification",
]
available = set(otb.Registry.GetAvailableApplications())
for name in NEEDED:
    print(f"{'OK     ' if name in available else 'MISSING'} {name}")

# Compare these keys with the ones used further down, and fix any mismatch.
for name in NEEDED:
    if name in available:
        app = otb.Registry.CreateApplication(name)
        print(f"\n{name}\n    {app.GetParametersKeys()}")

In [ ]:
# --- The common output grid: AOI bbox in UTM, snapped to PIXEL_SIZE ---
aoi_gs = gpd.GeoSeries([parse_polygon(AOI)], crs="EPSG:4326")
UTM_CRS = aoi_gs.estimate_utm_crs()
minx, miny, maxx, maxy = aoi_gs.to_crs(UTM_CRS).total_bounds

# Snapping outwards to round coordinates guarantees that every date, and any
# date added later, lands on exactly the same pixel centres.
ULX = np.floor(minx / PIXEL_SIZE) * PIXEL_SIZE
ULY = np.ceil(maxy / PIXEL_SIZE) * PIXEL_SIZE
SIZE_X = int(np.ceil((maxx - ULX) / PIXEL_SIZE))
SIZE_Y = int(np.ceil((ULY - miny) / PIXEL_SIZE))
TRANSFORM = from_origin(ULX, ULY, PIXEL_SIZE, PIXEL_SIZE)
EPSG = UTM_CRS.to_epsg()

print(f"{UTM_CRS.name} (EPSG:{EPSG})")
print(f"{SIZE_X} x {SIZE_Y} px at {PIXEL_SIZE} m, upper-left ({ULX}, {ULY})")

In [ ]:
def measurement_file(slc_path, swath, pol):
    """Path to the measurement GeoTIFF of one sub-swath and polarisation."""
    slc_path = Path(slc_path)
    if slc_path.suffix.lower() == ".zip":
        raise ValueError(f"OTB cannot read inside a .zip: extract {slc_path.name} first")
    pattern = f"s1?-{swath.lower()}-slc-{pol.lower()}-*.tiff"
    files = sorted((slc_path / "measurement").glob(pattern))
    if not files:
        raise FileNotFoundError(f"{pattern} not found in {slc_path}")
    return str(files[0])


def _run(app_name, params, out_key="out", out_path=None):
    """Run one OTB application and write its result to out_path."""
    app = otb.Registry.CreateApplication(app_name)
    for key, value in params.items():
        if isinstance(value, bool):
            app.SetParameterInt(key, int(value))
        elif isinstance(value, (list, tuple)):
            app.SetParameterStringList(key, [str(v) for v in value])
        elif isinstance(value, int):
            app.SetParameterInt(key, value)
        elif isinstance(value, float):
            app.SetParameterFloat(key, value)
        else:
            app.SetParameterString(key, str(value))
    app.SetParameterString(out_key, str(out_path))
    app.ExecuteAndWriteOutput()
    return str(out_path)


def swath_to_ortho(slc_path, swath, bursts, pol, tmp_dir):
    """Selected bursts of one sub-swath -> gamma0, orthorectified on the grid."""
    tmp = Path(tmp_dir)
    tmp.mkdir(parents=True, exist_ok=True)
    tag = f"{Path(slc_path).stem[:32]}_{swath}_{pol}"
    meas = measurement_file(slc_path, swath, pol)

    # 1. one file per selected burst — OTB indices are 0-based, ours are 1-based
    burst_files = [
        _run(
            "SARBurstExtraction",
            {"in": meas, "burstindex": b - 1, "allpixels": False},
            out_path=tmp / f"{tag}_burst{b}.tif",
        )
        for b in bursts
    ]

    # 2. glue them back into one continuous image
    concat = _run(
        "SARConcatenateBursts",
        {"il": burst_files, "insar": meas},
        out_path=tmp / f"{tag}_concat.tif",
    )

    # 3. radiometric calibration to gamma0 (ellipsoid LUT, not terrain-flattened)
    cal = _run(
        "SARCalibration",
        {"in": concat, "lut": "gamma", "removenoise": True},
        out_path=tmp / f"{tag}_gamma0.tif",
    )

    # 4. orthorectification onto the shared grid — identical for every date
    return _run(
        "OrthoRectification",
        {
            "io.in": cal,
            "map": "epsg",
            "map.epsg.code": EPSG,
            "outputs.mode": "outputroi",
            "outputs.ulx": ULX,
            "outputs.uly": ULY,
            "outputs.sizex": SIZE_X,
            "outputs.sizey": SIZE_Y,
            "outputs.spacingx": PIXEL_SIZE,
            "outputs.spacingy": -PIXEL_SIZE,
            "elev.dem": DEM_DIR,
            "elev.geoid": GEOID,
            "interpolator": "bco",
        },
        out_key="io.out",
        out_path=tmp / f"{tag}_ortho.tif",
    )

In [ ]:
def product_to_geotiff(slc_path, summary, out_dir, tmp_dir):
    """One product -> one 2-band GeoTIFF (gamma0_VV, gamma0_VH) on the grid."""
    bands = {}
    for pol in POLARISATIONS:
        layers = [
            swath_to_ortho(slc_path, swath, bursts, pol, tmp_dir)
            for swath, bursts in sorted(summary.items())
        ]
        # All layers share the grid, so they can be merged pixel by pixel:
        # average where sub-swaths overlap, keep the single value elsewhere.
        # Zero marks pixels outside the imaged area.
        total = np.zeros((SIZE_Y, SIZE_X), dtype="float64")
        count = np.zeros((SIZE_Y, SIZE_X), dtype="uint8")
        for path in layers:
            with rasterio.open(path) as src:
                values = src.read(1).astype("float64")
            valid = values > 0
            total[valid] += values[valid]
            count[valid] += 1
        bands[pol] = np.where(count > 0, total / np.maximum(count, 1), 0).astype("float32")

    out_path = Path(out_dir) / f"{Path(slc_path).stem}_gamma0.tif"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(
        out_path, "w", driver="GTiff",
        height=SIZE_Y, width=SIZE_X, count=len(POLARISATIONS),
        dtype="float32", crs=UTM_CRS, transform=TRANSFORM, nodata=0,
        compress="deflate", tiled=True,
    ) as dst:
        for index, pol in enumerate(POLARISATIONS, start=1):
            dst.write(bands[pol], index)
            dst.set_band_description(index, f"gamma0_{pol}")
    return out_path

In [ ]:
outputs = []
for slc in SLC_PATHS:
    path = product_to_geotiff(slc, selection[slc], OUT_DIR, TMP_DIR)
    print("written:", path)
    outputs.append(path)

In [ ]:
# --- Check the outputs: same grid everywhere, plausible gamma0 values ---
for path in outputs:
    with rasterio.open(path) as src:
        print(f"{Path(path).name}  {src.crs}  {src.width}x{src.height}")
        for index in range(1, src.count + 1):
            data = src.read(index)
            valid = data[data > 0]
            name = src.descriptions[index - 1]
            if valid.size:
                print(f"    {name}: {valid.size / data.size:.0%} valid, "
                      f"median {np.median(valid):.4f}, "
                      f"dB {10 * np.log10(np.median(valid)):.1f}")
            else:
                print(f"    {name}: empty")